# Building the building blocks of the transformer architecture

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

Let's see a single Head perform self-attention under the hood 

In [ ]:
torch.manual_seed(1337)

B, T, C = 4, 8, 32 # batch, time (context), channels (embedding size)
# input tensor after embedding the input tokens
X = torch.randn(B, T, C)

n_heads = 4
head_size = C // n_heads
key = nn.Linear(C, head_size, bias=False)
query = nn.Linear(C, head_size, bias=False)
value = nn.Linear(C, head_size, bias=False)
# all of them are comming from the same input X, so they are correlated
k = key(X)   # (B, T, head_size)
q = query(X) # (B, T, head_size)
v = value(X) # (B, T, head_size)

weights = q @ k.transpose(1, 2) * head_size**-0.5 # (B, T, T)
# mask out the lower triangle (causal attention)
weights = weights.masked_fill(torch.tril(torch.ones(T, T)) == 0, float('-inf'))
weights = F.softmax(weights, dim=-1) # (B, T, T)
out_h1 = weights @ v # (B, T, head_size)
# simulate multiple heads of self-attention in parallel
out_h2 = torch.randn_like(out_h1)
out_h3 = torch.randn_like(out_h1)
out_h4 = torch.randn_like(out_h1)
# concatenate the outputs of the heads
out = torch.cat([out_h1, out_h2, out_h3, out_h4], dim=-1) # (B, T, head_size * num_heads)
out = nn.Linear(head_size * n_heads, C, bias=False)(out) # (B, T, C)
# feedforward network
net = nn.Sequential(
    nn.Linear(C, C * 4),
    nn.ReLU(),
    nn.Linear(C * 4, C),
)
out = net(out) # (B, T, C)
out.shape

torch.Size([4, 8, 32])

Let's implement this in classes

In [ ]:
class Head(nn.Module):
    """ one head of self-attention """

    def __init__(self, n_embd, head_size):
        super().__init__()
        self.key = nn.Linear(n_embd, head_size, bias=False)
        self.query = nn.Linear(n_embd, head_size, bias=False)
        self.value = nn.Linear(n_embd, head_size, bias=False)
        
    def forward(self, x):
        # input of size (batch, time-step, channels)
        # output of size (batch, time-step, head size)
        k = self.key(x)   # (B, T, head_size)
        q = self.query(x) # (B, T, head_size)
        v = self.value(x) # (B, T, head_size)
        # compute attention scores ("affinities")
        weight = q @ k.transpose(1, 2) * head_size**-0.5 # (B, T, T)
        weight = weight.masked_fill(torch.tril(torch.ones(T, T)) == 0, float('-inf'))
        weight = F.softmax(weight, dim=-1) # (B, T, T)
        # perform the weighted aggregation of the values
        out = weight @ v # (B, T, head_size)
        return out
    
class MultiHeadAttention(nn.Module):
    """Multible heads of self-attention in parallel"""
    
    def __init__(self, n_embd, num_heads, head_size, dropout):
        super().__init__()
        self.heads = nn.ModuleList([Head(n_embd, head_size) for _ in range(num_heads)])
        self.proj = nn.Linear(num_heads * head_size, n_embd) 
        # dropout is applied before it is added to the sub-layer input (residual connection)
        self.dorpout = nn.Dropout(dropout)
        
    def forward(self, x):
        out = torch.cat([h(x) for h in self.heads], dim=-1)
        out = self.proj(out)
        out = self.dorpout(out)
        return out

class FeedForward(nn.Module):
    """A simple linear layer followed by a non-linearity"""
    
    def __init__(self, n_embd, dropout):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd),
            nn.ReLU(),
            nn.Linear(4 * n_embd, n_embd),
            nn.Dropout(dropout)
        )
        
    def forward(self, x):
        return self.net(x)

class Block(nn.Module):
    """ Transformer block: communication followed by computation """
    
    def __init__(self, n_embd, num_heads):
        super().__init__()
        head_size = n_embd // num_heads
        self.self_attn = MultiHeadAttention(n_embd, num_heads, head_size)
        self.ff = FeedForward(n_embd)
        self.layernorm1 = nn.LayerNorm(n_embd)
        self.layernorm2 = nn.LayerNorm(n_embd)
        
    # with residual connections, the input is added to the output of each sublayer
    # the layer normalization is applied before the addition
    def forward(self, x):
        x = x + self.self_attn(self.layernorm1(x))
        x = x + self.ff(self.layernorm2(x))
        return x

class GPT(nn.Module):
    """ The full GPT Language Model, with a stack of transformer blocks """
    
    def __init__(self, vocab_size, block_size, n_embd, num_heads, num_layers):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, n_embd)
        self.position_embedding_table = nn.Embedding(block_size, n_embd)
        # what the following line does is equivalent to:
        # self.blocks = nn.Sequential(
        #     Block(n_embd, num_heads),
        #     Block(n_embd, num_heads),
        #     Block(n_embd, num_heads),
        # )
        self.blocks = nn.Sequential(*[Block(n_embd, num_heads) for _ in range(num_layers)]) 
        self.ln_f = nn.LayerNorm(n_embd)
        self.lm_head = nn.Linear(n_embd, vocab_size)
        
    def forward(self, X):
        _, T = X.shape
        token_embeddings = self.token_embedding_table(X) # (B, T, C)
        position_embeddings = self.position_embedding_table(torch.arange(T)) # (T, C)
        x = token_embeddings + position_embeddings # (B, T, C)
        x = self.blocks(x) # (B, T, C)
        x = self.ln_f(x) # (B, T, C)
        logits = self.lm_head(x) # (B, T, vocab_size)
        return logits

torch.Size([4, 8, 5])